# 05 — Kaggle submission with XLM-RoBERTa large

## Want to see more?

I turned this Kaggle competition into a larger project, with notes on data exploration, training experiments, runtime comparisons, model evaluations, and dataset overlap. If you want to see how I approached it, come check out my [GitHub repository](https://github.com/KurtMeehanPro/adventure-of-the-lying-detective).

Now back to this notebook...

## Model choice and dataset leakage

While reviewing the leaderboard and public notebooks, I noticed that some high-scoring approaches used larger models, models previously fine-tuned on other natural-language inference datasets, or both.

My analysis found complete overlap between the Watson examples and examples in MNLI and XNLI. Because a model already fine-tuned on those datasets may have seen the same examples, I chose a general pretrained XLM-RoBERTa Large checkpoint and fine-tuned it using only the provided Watson training data.

For the evidence and full discussion, see my [leaderboard models and dataset leakage investigation](https://github.com/KurtMeehanPro/adventure-of-the-lying-detective/blob/main/docs/investigations/leaderboard-models-and-dataset-leakage.md).

## This notebook's approach

I will fine-tune `FacebookAI/xlm-roberta-large` with its matching tokenizer for three-way natural-language inference and create predictions for Kaggle's test set.

The loading, batching, training, and submission code is self-contained for Kaggle.

## Model and tokenizer choice

- starting from general pretrained [XLM-RoBERTa large checkpoint](https://huggingface.co/FacebookAI/xlm-roberta-large)
- newly initialized three-label classification head. 
- The tokenizer receives premise first, hypothesis second, with dynamic batch padding and configurable truncation.

In [ ]:
from pathlib import Path
import pandas as pd

KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/contradictory-my-dear-watson")


def load_datasets():
    """Load the competition data attached to this Kaggle notebook."""
    files = {
        "train": KAGGLE_DATA_DIR / "train.csv",
        "test": KAGGLE_DATA_DIR / "test.csv",
        "sample_submission": KAGGLE_DATA_DIR / "sample_submission.csv",
    }
    missing_files = [path for path in files.values() if not path.is_file()]
    if missing_files:
        missing = ", ".join(str(path) for path in missing_files)
        raise FileNotFoundError(
            f"Expected Kaggle competition files were not found: {missing}"
        )
    return {name: pd.read_csv(path) for name, path in files.items()}

## Bring in the data

In [ ]:
datasets = load_datasets()

training_data = datasets["train"].copy()
test_data = datasets["test"].copy()

## Data Cleaning

Prior exploration in [Notebook 1](https://github.com/KurtMeehanPro/adventure-of-the-lying-detective/blob/main/notebooks/01_dataset_exploration.ipynb) found

- no missing values
- no duplicate premise–hypothesis pairs

No further text cleaning is planned; the original premise and hypothesis text will be kept.

## Bring in Pre-Trained Model

### XLM-Roberta Large is chosen as the pre-trained model.

In [ ]:
# Run before model initialization and loader creation for each fresh experiment.
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer


def load_model_and_tokenizer(model_name=None):
    """Load a pretrained model with a three-label head and its matching tokenizer."""
    if model_name is None:
        raise ValueError("Model name is required when calling load_model_and_tokenizer.")

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            model_name, clean_up_tokenization_spaces=False)
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=3)
    except OSError as error:
        raise RuntimeError(f"Failed to load the model or tokenizer for {model_name}.") from error
    return model, tokenizer

In [ ]:
model, tokenizer = load_model_and_tokenizer("FacebookAI/xlm-roberta-large")

## Tokenization

In [ ]:
def tokenize_data(data, tokenizer):
    """Tokenize premise–hypothesis pairs without padding, retaining their labels."""
    encodings = tokenizer(
        data["premise"].tolist(),
        data["hypothesis"].tolist(),
        truncation=True,
        padding=False,
    )
    encodings["labels"] = data["label"].tolist()
    return encodings

In [ ]:
training_encodings = tokenize_data(training_data, tokenizer)

## Batching, Padding, Training Settings

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorWithPadding


class EncodedDataset(Dataset):
    """Expose one encoded example and its label at a time."""

    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["labels"])

    def __getitem__(self, index):
        return {key: values[index] for key, values in self.encodings.items()}


def make_loaders(training_encodings, tokenizer, batch_size):
    """Pad each batch to its longest example and return PyTorch tensors."""
    collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")
    training_loader = DataLoader(
        EncodedDataset(training_encodings), batch_size=batch_size,
        shuffle=True, collate_fn=collator,
    )

    return training_loader

In [ ]:
batch_size = 16
learning_rate = 7.5e-6
epochs = 5
weight_decay = 0.01  # AdamW default: 0.01.

In [ ]:
training_loader = make_loaders(training_encodings, tokenizer, batch_size)

## Fine-tuning

In [ ]:
from datetime import datetime

from transformers import get_constant_schedule_with_warmup

def fine_tune(model, training_loader, learning_rate, epochs, weight_decay=0.01):
    """Fine-tune the model for a fixed number of epochs with linear warmup.

    Start at zero and reach learning_rate after the first 10% of optimizer steps
    (rounded down, minimum one step), with no learning-rate decay afterward.

    """
    if epochs < 1 or len(training_loader) == 0:
        raise ValueError("Training needs positive epochs and nonempty loaders")

    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available()
        else "cpu"
    )
    model.to(device)
    if device.type == "cuda" and torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    total_steps = len(training_loader) * epochs
    warmup_steps = max(1, int(total_steps * 0.1))
    scheduler = get_constant_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
    )

    for epoch in range(epochs):
        print(
            f"{datetime.now():%Y-%m-%d %H:%M:%S} | "
            f"Starting epoch {epoch + 1}/{epochs}",
            flush=True,
        )
        model.train()
        total_loss = 0.0
        total_examples = 0

        for batch_idx, batch in enumerate(training_loader, start=1):
            if batch_idx % 250 == 0:
                print(
                    f"{datetime.now():%Y-%m-%d %H:%M:%S} | "
                    f"Processing batch {batch_idx}/{len(training_loader)}",
                    flush=True,
                )

            ## Move the batch to the same device as the model
            batch = {key: value.to(device) for key, value in batch.items()}

            optimizer.zero_grad()
            loss = model(**batch).loss.mean()
            loss.backward()
            optimizer.step()
            scheduler.step()
            batch_size = batch["labels"].size(0)
            total_loss += loss.item() * batch_size
            total_examples += batch_size

        training_loss = total_loss / total_examples
        print(
            f"{datetime.now():%Y-%m-%d %H:%M:%S} | "
            f"Epoch {epoch + 1}/{epochs} - training loss: {training_loss:.4f}",
            flush=True,
        )

    model.eval()

    return model


In [ ]:
model = fine_tune(
    model, training_loader, learning_rate, epochs, weight_decay=weight_decay
    )

## Kaggle submission

The predictions are written in the same row order and schema as `sample_submission.csv`.

### Prepare the test loader

In [ ]:
test_encodings = tokenizer(
    test_data["premise"].tolist(),
    test_data["hypothesis"].tolist(),
    truncation=True,
    padding=False,
)

class TestDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        return {key: values[index] for key, values in self.encodings.items()}

test_loader = DataLoader(
    TestDataset(test_encodings),
    batch_size=batch_size,
    shuffle=False,
    collate_fn=DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt"),
)

### Predict the test labels

In [ ]:
test_predictions = []
device = next(model.parameters()).device

model.eval()
with torch.no_grad():
    for batch in test_loader:
        batch = {key: value.to(device) for key, value in batch.items()}
        logits = model(**batch).logits
        test_predictions.extend(logits.argmax(dim=-1).cpu().tolist())

if len(test_predictions) != len(test_data):
    raise ValueError("Test predictions and test examples are not aligned.")

### Write `submission.csv`

In [ ]:
submission = datasets["sample_submission"].copy()
if not submission["id"].equals(test_data["id"]):
    raise ValueError("Sample-submission IDs and test IDs are not aligned.")
submission["prediction"] = test_predictions

submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)

display(submission.head())
print(f"Wrote {len(submission):,} predictions to {submission_path}")